# Chapter 7 Project — Bengaluru Hospital Readmission Cost Prediction

**Goal:** Predict the total cost (₹) of a patient being readmitted to a hospital within 30 days.

**Why this problem?**
Hospitals file insurance claims based on predicted readmission costs. If the prediction is wrong:
- Too high → insurance company disputes the claim
- Too low → hospital loses money

Accurate prediction matters. And accurate prediction requires well-tuned models — not guessed hyperparameters.

**What this project demonstrates:**
Everything from Chapter 7 — RandomizedSearchCV + Pipeline — applied end-to-end on a realistic problem with mixed features (numeric + categorical), then two algorithms compared properly after tuning.

---

## The Workflow

```
Raw Data
  → EDA (understand the problem)
  → Preprocessing (encode categoricals, handle features)
  → Baseline (default hyperparameters — benchmark)
  → RandomizedSearchCV + Pipeline → Best RandomForest
  → RandomizedSearchCV + Pipeline → Best XGBoost
  → Compare → Declare winner
```

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import uniform, randint
from xgboost import XGBRegressor

np.random.seed(42)

## Step 1 — Build the Dataset

**Features:**
- `age` — patient age in years
- `gender` — Male / Female
- `diagnosis` — primary diagnosis category
- `prev_admissions` — number of previous hospital admissions
- `length_of_stay` — days spent in hospital during first admission
- `num_medications` — number of medications prescribed on discharge
- `hba1c` — blood sugar control marker (higher = worse, especially for diabetics)
- `creatinine` — kidney function marker (higher = worse)
- `attended_followup` — did patient attend the follow-up consultation? (0=No, 1=Yes)

**Target:** `readmission_cost` in ₹

In [ ]:
n = 1200

age               = np.random.randint(25, 85, n)
gender            = np.random.choice(['Male', 'Female'], n)
diagnosis         = np.random.choice(['Diabetes', 'Cardiac', 'Respiratory', 'Renal', 'Orthopaedic'], n)
prev_admissions   = np.random.randint(0, 8, n)
length_of_stay    = np.random.randint(2, 20, n)
num_medications   = np.random.randint(2, 15, n)
hba1c             = np.random.uniform(4.5, 12.0, n)
creatinine        = np.random.uniform(0.6, 4.5, n)
attended_followup = np.random.randint(0, 2, n)

# Diagnosis multiplier — some conditions cost more to treat
diag_cost = {'Diabetes': 1.0, 'Cardiac': 1.8, 'Respiratory': 1.3, 'Renal': 1.6, 'Orthopaedic': 1.4}
diag_multiplier = np.array([diag_cost[d] for d in diagnosis])

# Readmission cost — driven by multiple factors
readmission_cost = (
    5000 * diag_multiplier                          # base cost by diagnosis
    + 800  * length_of_stay                         # longer stay = higher cost
    + 1200 * prev_admissions                        # more history = more complex case
    + 400  * num_medications                        # more meds = more cost
    + 600  * creatinine                             # poor kidney function = expensive
    + 300  * hba1c                                  # poor blood sugar control = expensive
    + 200  * age                                    # older patients cost more
    - 3000 * attended_followup                      # followup reduces readmission severity
    + np.random.normal(0, 1500, n)                  # real-world noise
).clip(min=3000)                                    # floor at ₹3000 — no negative costs

df = pd.DataFrame({
    'age':               age,
    'gender':            gender,
    'diagnosis':         diagnosis,
    'prev_admissions':   prev_admissions,
    'length_of_stay':    length_of_stay,
    'num_medications':   num_medications,
    'hba1c':             hba1c,
    'creatinine':        creatinine,
    'attended_followup': attended_followup,
    'readmission_cost':  readmission_cost
})

print("Dataset shape:", df.shape)
df.head()

## Step 2 — EDA

In [ ]:
print(df.describe().round(2))

In [ ]:
# Distribution of target variable
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(df['readmission_cost'], bins=40, color='steelblue', edgecolor='white')
plt.xlabel('Readmission Cost (₹)')
plt.ylabel('Count')
plt.title('Distribution of Readmission Cost')

plt.subplot(1, 2, 2)
avg_cost_by_diag = df.groupby('diagnosis')['readmission_cost'].mean().sort_values(ascending=False)
avg_cost_by_diag.plot(kind='bar', color='coral', edgecolor='white')
plt.xlabel('Diagnosis')
plt.ylabel('Avg Cost (₹)')
plt.title('Average Cost by Diagnosis')
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

## Step 3 — Preprocessing

We have two types of features:
- **Numeric:** age, prev_admissions, length_of_stay, num_medications, hba1c, creatinine, attended_followup → StandardScaler
- **Categorical:** gender, diagnosis → OneHotEncoder

We use `ColumnTransformer` to apply different preprocessing to different columns — all inside the Pipeline.

**Why OneHotEncoder for categoricals?**
Models can't work with text like 'Cardiac' or 'Male'. OneHotEncoder converts each category into a binary column:
- `diagnosis_Cardiac`, `diagnosis_Diabetes`, `diagnosis_Renal`... (1 if that diagnosis, 0 otherwise)
- `gender_Male`, `gender_Female` (1 if that gender, 0 otherwise)

**Why ColumnTransformer?**
You can't scale text columns. You can't OneHotEncode numeric columns. ColumnTransformer lets you apply the right transformation to the right columns in one step — and it works cleanly inside a Pipeline.

In [ ]:
X = df.drop('readmission_cost', axis=1)
y = df['readmission_cost']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Identify column types
numeric_features     = ['age', 'prev_admissions', 'length_of_stay',
                         'num_medications', 'hba1c', 'creatinine', 'attended_followup']
categorical_features = ['gender', 'diagnosis']

# ColumnTransformer: applies the right preprocessor to the right columns
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),          # scale numeric columns
    ('cat', OneHotEncoder(drop='first'), categorical_features)  # encode categorical columns
                                                          # drop='first' avoids dummy variable trap
])

print("Numeric features: ", numeric_features)
print("Categorical features:", categorical_features)
print("Train size:", X_train.shape)
print("Test size: ", X_test.shape)

## Step 4 — Baseline (Default Hyperparameters)

Always benchmark first. This tells us how much tuning actually helps.

In [ ]:
def evaluate(name, y_true, y_pred):
    """Print RMSE, MAE, R² for any model. Used throughout this notebook."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"{name}")
    print(f"  RMSE : ₹{rmse:,.0f}")
    print(f"  MAE  : ₹{mae:,.0f}")
    print(f"  R²   : {r2:.4f}")
    return rmse, mae, r2

# Baseline RandomForest — default hyperparameters, inside Pipeline
baseline_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])
baseline_pipe.fit(X_train, y_train)
y_pred_base = baseline_pipe.predict(X_test)

rmse_base, mae_base, r2_base = evaluate("Baseline RandomForest (default)", y_test, y_pred_base)

## Step 5 — Tune RandomForest with RandomizedSearchCV + Pipeline

### Why RandomizedSearch (not Grid) here?
We're searching over 5 hyperparameters. If we used GridSearchCV with 5 values each: 5⁵ = 3,125 combinations × 5 folds = 15,625 fits. Too slow.

RandomizedSearch with `n_iter=60`: 60 × 5 = 300 fits. Fast — and finds a very good combination.

### Why Pipeline?
Without Pipeline: if you preprocess before GridSearch, the validation fold's statistics leak into the preprocessor. The CV scores look better than real-world performance — you're cheating without knowing it.

With Pipeline: preprocessor fits only on training folds. Validation fold is truly unseen. CV scores are honest.

In [ ]:
# --- RandomForest Pipeline ---
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

# param name = 'model__hyperparameter_name'
# 'model__' prefix tells RandomizedSearch: this param belongs to the 'model' step
rf_param_dist = {
    'model__n_estimators':      randint(100, 500),   # number of trees
    'model__max_depth':         [5, 10, 15, 20, None],
    'model__min_samples_split': randint(2, 20),      # min samples to split a node
    'model__min_samples_leaf':  randint(1, 10),      # min samples in a leaf
    'model__max_features':      uniform(0.3, 0.7)    # fraction of features per split
}

rf_random = RandomizedSearchCV(
    estimator           = rf_pipe,
    param_distributions = rf_param_dist,
    n_iter              = 60,                        # try 60 random combinations
    cv                  = 5,
    scoring             = 'neg_mean_squared_error',
    n_jobs              = -1,
    random_state        = 42,
    verbose             = 1
)

rf_random.fit(X_train, y_train)                     # raw X_train — Pipeline handles preprocessing
print("\nRandomForest search complete!")

In [ ]:
print("Best RF Hyperparameters:")
for k, v in rf_random.best_params_.items():
    print(f"  {k}: {v}")

best_rf_cv_rmse = np.sqrt(-rf_random.best_score_)
print(f"\nBest CV RMSE: ₹{best_rf_cv_rmse:,.0f}")

y_pred_rf = rf_random.best_estimator_.predict(X_test)
rmse_rf, mae_rf, r2_rf = evaluate("\nTuned RandomForest", y_test, y_pred_rf)

## Step 6 — Tune XGBoost with RandomizedSearchCV + Pipeline

XGBoost has more hyperparameters than RandomForest — GridSearchCV would be even more impractical here. RandomizedSearch is the right tool.

**Key XGBoost hyperparameters being tuned:**

| Hyperparameter | What it controls |
|---|---|
| `n_estimators` | Number of trees (boosting rounds) |
| `max_depth` | How deep each tree grows |
| `learning_rate` | How much each tree contributes (η) — lower = more careful |
| `subsample` | Fraction of training rows used per tree — prevents overfitting |
| `colsample_bytree` | Fraction of features used per tree — prevents overfitting |
| `reg_alpha` | L1 regularisation on leaf weights |
| `reg_lambda` | L2 regularisation on leaf weights |

In [ ]:
# --- XGBoost Pipeline ---
xgb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(random_state=42, verbosity=0))
])

xgb_param_dist = {
    'model__n_estimators':     randint(100, 500),
    'model__max_depth':        randint(3, 10),
    'model__learning_rate':    uniform(0.01, 0.29),   # range: 0.01 to 0.30
    'model__subsample':        uniform(0.6, 0.4),     # range: 0.6 to 1.0
    'model__colsample_bytree': uniform(0.6, 0.4),     # range: 0.6 to 1.0
    'model__reg_alpha':        uniform(0, 1),         # L1 regularisation
    'model__reg_lambda':       uniform(0.5, 2)        # L2 regularisation
}

xgb_random = RandomizedSearchCV(
    estimator           = xgb_pipe,
    param_distributions = xgb_param_dist,
    n_iter              = 60,
    cv                  = 5,
    scoring             = 'neg_mean_squared_error',
    n_jobs              = -1,
    random_state        = 42,
    verbose             = 1
)

xgb_random.fit(X_train, y_train)
print("\nXGBoost search complete!")

In [ ]:
print("Best XGBoost Hyperparameters:")
for k, v in xgb_random.best_params_.items():
    print(f"  {k}: {v}")

best_xgb_cv_rmse = np.sqrt(-xgb_random.best_score_)
print(f"\nBest CV RMSE: ₹{best_xgb_cv_rmse:,.0f}")

y_pred_xgb = xgb_random.best_estimator_.predict(X_test)
rmse_xgb, mae_xgb, r2_xgb = evaluate("\nTuned XGBoost", y_test, y_pred_xgb)

## Step 7 — Final Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Baseline RF (default)', 'Tuned RandomForest', 'Tuned XGBoost'],
    'RMSE (₹)': [rmse_base, rmse_rf, rmse_xgb],
    'MAE (₹)':  [mae_base,  mae_rf,  mae_xgb],
    'R²':       [r2_base,   r2_rf,   r2_xgb]
})

results['RMSE (₹)'] = results['RMSE (₹)'].apply(lambda x: f"₹{x:,.0f}")
results['MAE (₹)']  = results['MAE (₹)'].apply(lambda x: f"₹{x:,.0f}")
results['R²']       = results['R²'].apply(lambda x: f"{x:.4f}")

print(results.to_string(index=False))

In [ ]:
# --- Visual Comparison: Actual vs Predicted for both tuned models ---

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_pred, title, color in zip(
    axes,
    [y_pred_rf, y_pred_xgb],
    ['Tuned RandomForest', 'Tuned XGBoost'],
    ['steelblue', 'coral']
):
    ax.scatter(y_test, y_pred, alpha=0.35, color=color, edgecolors='white', linewidth=0.3)
    ax.plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()],
            'k--', linewidth=1.5, label='Perfect prediction')
    ax.set_xlabel('Actual Cost (₹)')
    ax.set_ylabel('Predicted Cost (₹)')
    ax.set_title(title)
    ax.legend()

plt.suptitle('Bengaluru Hospital — Readmission Cost: Actual vs Predicted', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Summary

### What This Project Demonstrated

| Step | What we did | Why |
|---|---|---|
| Baseline | Default hyperparameters | Benchmark — measure improvement from tuning |
| ColumnTransformer | Scale numeric, encode categorical | Different columns need different preprocessing |
| Pipeline | Chain preprocessor + model | Prevent data leakage during CV |
| RandomizedSearchCV | Sample 60 random combinations | Too many hyperparameters for GridSearch |
| Refit | Automatic — on all training data | Use 100% of data for final model |
| Compare | RF vs XGBoost after proper tuning | Fair comparison — both tuned, not just defaulted |

### Key Lesson

Comparing algorithms at **default hyperparameters** is not a fair comparison. One algorithm might just happen to have better defaults for your data. Always tune both before declaring a winner.

### New Concept Introduced: ColumnTransformer

When your data has both numeric and categorical columns, you need different preprocessing for each. `ColumnTransformer` applies:
- `StandardScaler` → numeric columns
- `OneHotEncoder` → categorical columns

All inside the Pipeline — so leakage prevention works for both transformations.